## Bike Sharing Dataset – Registros Diarios

El conjunto de datos de Bike Sharing contiene registros diarios del sistema de bicicletas compartidas Capital Bikeshare en Washington D.C. (2011–2012). Incluye información meteorológica, calendario y conteos de usuarios con el objetivo de predecir la demanda de alquiler de bicicletas.

**Dataset:** [Bike Sharing Dataset – UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

---

| Columna (índice) | Nombre en el CSV | Descripción |
|-----------------|-----------------|-------------|
| 0 | <span style="color:red">instant</span> | Índice de registro (eliminado) |
| 1 | <span style="color:red">dteday</span> | Fecha (eliminada) |
| 2 | `season` | Estación del año (1=invierno, 2=primavera, 3=verano, 4=otoño) |
| 3 | `yr` | Año (0=2011, 1=2012) |
| 4 | `mnth` | Mes (1–12) |
| 5 | `holiday` | Día festivo (0=no, 1=sí) |
| 6 | `weekday` | Día de la semana (0=domingo … 6=sábado) |
| 7 | `workingday` | Día laboral (0=no, 1=sí) |
| 8 | `weathersit` | Situación meteorológica (1=despejado, 2=nublado, 3=lluvia/nieve leve) |
| 9 | `temp` | Temperatura normalizada (°C) |
| 10 | `atemp` | Sensación térmica normalizada (°C) |
| 11 | `hum` | Humedad normalizada |
| 12 | `windspeed` | Velocidad del viento normalizada |
| 13 | <span style="color:red">casual</span> | Usuarios no registrados (eliminado – componente de cnt) |
| 14 | <span style="color:red">registered</span> | Usuarios registrados (eliminado – componente de cnt) |
| 15 | `cnt` | **Variable objetivo:** total de bicicletas alquiladas por día |

**Variable objetivo (col. 15):** `cnt` (total de alquileres diarios)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## 2. Carga y Selección de Datos

Se carga el archivo CSV y se eliminan columnas que no aportan valor predictivo:
- **`instant`** y **`dteday`**: identificadores sin valor predictivo.
- **`casual`** y **`registered`**: son componentes directas de `cnt` (variable objetivo), incluirlas causaría *data leakage*.

In [2]:
dt = pd.read_csv('../Database/PP4_day.csv')
dt.columns = dt.columns.str.strip()

columnas_a_excluir = ['instant', 'dteday', 'casual', 'registered']
dt = dt.drop(columns=columnas_a_excluir)

print("Columnas seleccionadas:")
print(dt.columns.tolist())
print(f"\nDimensiones del dataset: {dt.shape}")

Columnas seleccionadas:
['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']

Dimensiones del dataset: (731, 12)


## 3. Diagnóstico inicial de calidad de datos

Antes de limpiar, inspeccionamos el estado del dataset:
- Valores nulos (`NaN` / `None`)
- Ceros problemáticos en variables continuas (humedad, temperatura, velocidad del viento)
- Tipos de datos

In [3]:
print("=" * 50)
print("TIPOS DE DATOS")
print("=" * 50)
print(dt.dtypes)

print("\n" + "=" * 50)
print("VALORES NULOS POR COLUMNA")
print("=" * 50)
nulls = dt.isnull().sum()
print(nulls)
print(f"\nTotal de valores nulos: {nulls.sum()}")

print("\n" + "=" * 50)
print("CEROS EN VARIABLES CONTINUAS (posibles valores inválidos)")
print("=" * 50)
# Solo revisamos continuas donde un 0 es físicamente inválido
continuas = ['temp', 'atemp', 'hum', 'windspeed']
for col in continuas:
    n_ceros = (dt[col] == 0).sum()
    print(f"  {col}: {n_ceros} ceros")

print("\n" + "=" * 50)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("=" * 50)
print(dt.describe())

TIPOS DE DATOS
season          int64
yr              int64
mnth            int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
cnt             int64
dtype: object

VALORES NULOS POR COLUMNA
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
cnt           0
dtype: int64

Total de valores nulos: 0

CEROS EN VARIABLES CONTINUAS (posibles valores inválidos)
  temp: 0 ceros
  atemp: 0 ceros
  hum: 1 ceros
  windspeed: 0 ceros

ESTADÍSTICAS DESCRIPTIVAS
           season          yr        mnth     holiday     weekday  workingday  \
count  731.000000  731.000000  731.000000  731.000000  731.000000  731.000000   
mean     2.496580    0.500684    6.519836    0.028728    2.997264    0.683995   
std      1.110807    0.500342    3.451913    0.167

## 4. Limpieza de datos

### Estrategia:
1. **Ceros en `hum`**: La humedad normalizada igual a 0 es físicamente imposible → imputar con KNN (mediana de los 5 vecinos más cercanos basados en columnas numéricas).
2. **Ceros en variables categóricas ordinales** (`yr`, `holiday`, `weekday`, `workingday`): Son valores codificados válidos (ej. año 0 = 2011, domingo = weekday 0), **NO se imputan**.
3. **No hay valores NaN** en este dataset → no se requiere imputación adicional.

In [ ]:
numeric_cols = dt.select_dtypes(include=[np.number]).columns.tolist()

# --- KNN para imputar ceros inválidos en variables continuas ---
# Variables donde un 0 es físicamente inválido
continuas_invalidas = ['hum']  # temp/atemp/windspeed=0 es teóricamente posible

# Verificar si realmente hay ceros problemáticos
ceros_invalidos = {col: (dt[col] == 0).sum() for col in continuas_invalidas}
print("Ceros inválidos detectados:")
for col, n in ceros_invalidos.items():
    print(f"  {col}: {n}")

total_invalidos = sum(ceros_invalidos.values())

if total_invalidos > 0:
    print("\nAplicando imputación KNN sobre ceros inválidos...")

    # Escalar para que las distancias sean comparables
    scaler = StandardScaler()
    # Usamos solo filas SIN ceros inválidos para ajustar el scaler y el KNN
    mask_valido = pd.Series([True] * len(dt))
    for col in continuas_invalidas:
        mask_valido = mask_valido & (dt[col] != 0)

    X_num_all = scaler.fit_transform(dt[numeric_cols])

    knn = NearestNeighbors(n_neighbors=6, metric='euclidean')  # 6: incluye la fila misma
    knn.fit(X_num_all)
    distancias, indices = knn.kneighbors(X_num_all)

    for col in continuas_invalidas:
        mask_cero = dt[col] == 0
        n_ceros = mask_cero.sum()
        if n_ceros == 0:
            continue
        print(f"  Imputando {n_ceros} cero(s) en '{col}' con mediana KNN...")
        filas_cero = dt.index[mask_cero].tolist()
        for fila in filas_cero:
            vecinos_idx = indices[fila][1:6]  # excluir la fila misma (índice 0)
            valores_vecinos = dt[col].iloc[vecinos_idx]
            valores_vecinos = valores_vecinos[valores_vecinos != 0]  # excluir también si vecino=0
            if len(valores_vecinos) > 0:
                dt.at[fila, col] = valores_vecinos.median()
            else:
                # fallback: mediana global excluyendo ceros
                dt.at[fila, col] = dt[col][dt[col] != 0].median()

    print("\n✓ Imputación KNN completada.")
else:
    print("\n✓ No se detectaron ceros inválidos en variables continuas.")

print(f"\n¿Quedan NaN en el dataset? {dt.isnull().sum().sum()}")
print(f"¿Quedan ceros en 'hum'?     {(dt['hum'] == 0).sum()}")

## 5. Verificación final del dataset limpio

In [ ]:
print("=" * 50)
print("RESUMEN FINAL")
print("=" * 50)
print(f"Filas:    {dt.shape[0]}")
print(f"Columnas: {dt.shape[1]}")
print(f"NaN totales: {dt.isnull().sum().sum()}")

print("\nTipos de datos finales:")
print(dt.dtypes.value_counts())

print("\nPrimeras 5 filas del dataset preparado:")
print(dt.head())

## 6. Visualización exploratoria

Distribución de la variable objetivo `cnt` y correlación con variables numéricas.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Análisis exploratorio – Bike Sharing (día)', fontsize=14, fontweight='bold')

# 1. Distribución de cnt
axes[0, 0].hist(dt['cnt'], bins=30, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribución de cnt (alquileres/día)')
axes[0, 0].set_xlabel('cnt')
axes[0, 0].set_ylabel('Frecuencia')

# 2. cnt vs temperatura
axes[0, 1].scatter(dt['temp'], dt['cnt'], alpha=0.4, color='tomato', s=15)
axes[0, 1].set_title('cnt vs Temperatura')
axes[0, 1].set_xlabel('temp (normalizado)')
axes[0, 1].set_ylabel('cnt')

# 3. cnt vs humedad
axes[0, 2].scatter(dt['hum'], dt['cnt'], alpha=0.4, color='mediumseagreen', s=15)
axes[0, 2].set_title('cnt vs Humedad')
axes[0, 2].set_xlabel('hum (normalizado)')
axes[0, 2].set_ylabel('cnt')

# 4. cnt por estación
season_labels = {1: 'Invierno', 2: 'Primavera', 3: 'Verano', 4: 'Otoño'}
season_means = dt.groupby('season')['cnt'].mean()
axes[1, 0].bar([season_labels[s] for s in season_means.index], season_means.values,
               color=['#AED6F1', '#A9DFBF', '#F9E79F', '#F0B27A'])
axes[1, 0].set_title('Promedio de cnt por Estación')
axes[1, 0].set_ylabel('cnt promedio')

# 5. cnt por mes
month_means = dt.groupby('mnth')['cnt'].mean()
axes[1, 1].plot(month_means.index, month_means.values, marker='o', color='purple')
axes[1, 1].set_title('Promedio de cnt por Mes')
axes[1, 1].set_xlabel('Mes')
axes[1, 1].set_ylabel('cnt promedio')
axes[1, 1].set_xticks(range(1, 13))

# 6. Mapa de calor de correlaciones
import matplotlib.colors as mcolors
corr = dt.corr(numeric_only=True)
im = axes[1, 2].imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[1, 2].set_xticks(range(len(corr.columns)))
axes[1, 2].set_yticks(range(len(corr.columns)))
axes[1, 2].set_xticklabels(corr.columns, rotation=90, fontsize=7)
axes[1, 2].set_yticklabels(corr.columns, fontsize=7)
axes[1, 2].set_title('Correlaciones')
plt.colorbar(im, ax=axes[1, 2], shrink=0.8)

plt.tight_layout()
plt.savefig('PP3_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada como PP3_exploratorio.png")

## 7. Exportar dataset limpio

Se exporta el dataset preparado para su uso en las siguientes prácticas (modelado).

In [ ]:
dt.to_csv('PP3_day_clean.csv', index=False)
print("✓ Dataset limpio exportado como 'PP3_day_clean.csv'")
print(f"  Filas: {dt.shape[0]} | Columnas: {dt.shape[1]}")
print(f"  NaN totales: {dt.isnull().sum().sum()}")
print(f"  Ceros en 'hum': {(dt['hum'] == 0).sum()}")

In [4]:
import pandas as pd
import numpy as np

column_names = [
    'sample_code', 'clump_thickness', 'cell_size_uniformity', 
    'cell_shape_uniformity', 'marginal_adhesion', 'single_cell_size',
    'bare_nuclei', 'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class'
]

dt = pd.read_csv('Database/PP8_breast-cancer-wisconsin.data',
    header=None,
    names=column_names,
    na_values='?',
    skipinitialspace=True
)

print('=== Análisis de datos ===')
print(f'Total filas: {len(dt)}')
print(f'Total columnas: {len(dt.columns)}')
print()
print('Valores NaN por columna:')
print(dt.isnull().sum())
print()
print('Valores 0 por columna (numéricas):')
for col in dt.select_dtypes(include=[np.number]).columns:
    ceros = (dt[col] == 0).sum()
    if ceros > 0:
        print(f'  {col}: {ceros} ceros')
print()
print('Valores únicos en bare_nuclei (columna problemática conocida):')
print(dt['bare_nuclei'].unique())

FileNotFoundError: [Errno 2] No such file or directory: 'Database/PP8_breast-cancer-wisconsin.data'